# 1. Data Exploration & Pipeline Overview

**Purpose:** Full data loading → summary statistics → visualisation pipeline.

**Covers:**
1. Loading data (real CSVs, saved pkl, or synthetic)
2. Data class hierarchy inspection
3. Session-level summaries and feature matrix
4. Psychometric curves
5. Stat trajectories across sessions
6. Update matrices and serial dependence profiles
7. Manipulation detection and shift-aligned analysis

In [ ]:
%matplotlib inline

import os, sys
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# behav_utils — data
from behav_utils.data.structures import ExperimentData, AnimalData, SessionData
from behav_utils.data.synthetic import (
    generate_synthetic_animal,
    generate_synthetic_session,
    noisy_psychometric_simulator,
    sample_stimuli,
)
from behav_utils.data.loading import load_experiment, load_from_directory

# behav_utils — analysis
from behav_utils.analysis.summary_stats import (
    compute_summary_stats,
    list_available_stats,
    FEATURE_MATRIX_STATS,
    DEFAULT_STATS,
)
from behav_utils.analysis.session_features import (
    build_feature_matrix,
    build_feature_matrix_multi,
    summarise_features,
)
from behav_utils.analysis.update_matrix import compute_update_matrix
from behav_utils.analysis.utils import cumulative_gaussian

# behav_utils — plotting
from behav_utils.plotting.psychometric import (
    plot_psychometric,
    plot_session_psychometrics,
)
from behav_utils.plotting.trajectory import (
    plot_stat_trajectory,
    plot_multi_animal_trajectory,
    plot_stat_grid,
)
from behav_utils.plotting.update_matrix import (
    plot_update_matrix,
    plot_conditional_psychometrics,
)

print("All imports OK.")
print(f"Available summary statistics ({len(list_available_stats())}):")
for s in list_available_stats():
    print(f"  - {s}")

---
## 1. Load Data

Three options — uncomment the one you need.

**Option A:** `load_experiment('config.yaml')` — reads CSVs from disk using your config.  
**Option B:** `ExperimentData.load('experiment.pkl')` — reload from a saved pickle.  
**Option C:** Generate synthetic data (default for pipeline testing).

### Option A
`load_experiment('config.yaml')` — reads CSVs from disk using your config.  


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION A: Load from CSV files
# ═══════════════════════════════════════════════════════════════════════════

# path_config = 'path/to/config.yaml'
# experiment = load_experiment(path_config)
# STAGE = 'Full_Task_Cont'
# SHIFT_SESSION = None

# # Or point directly at a data directory:
# data_path = ''
# experiment = load_from_directory(data_path, config_path=path_config)

# # Load a single animal:
# from behav_utils.data.loading import load_animal
# from behav_utils.config.schema import load_config
# config = load_config(config_path)
# animal_path = ''
# animal = load_animal(animal_path, config)


### Option B
`ExperimentData.load('experiment.pkl')` — reload from a saved pickle.  


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION B: Load from saved pickle
# ═══════════════════════════════════════════════════════════════════════════
# experiment = ExperimentData.load('experiment.pkl')
# animal = AnimalData.load('SS05.pkl')

### Option C
Generate synthetic data (default for pipeline testing).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION C: Generate synthetic data (default)
# ═══════════════════════════════════════════════════════════════════════════
USE_SYNTHETIC = True

if USE_SYNTHETIC:
    def _learning_simulator(stimuli, categories, rng, sigma=0.3, lapse=0.05, **kw):
        from scipy.stats import norm
        p_b = lapse + (1 - 2 * lapse) * norm.cdf(stimuli, 0, sigma)
        return (rng.random(len(stimuli)) < p_b).astype(float)

    def generate_cohort(n_animals=5, n_sessions=25, shift_session=15, seed=42):
        rng = np.random.default_rng(seed)
        experiment = ExperimentData(metadata={'cohort': 'synthetic_demo'})
        for a in range(n_animals):
            animal_id = f'SYN{a+1:02d}'
            animal_seed = int(rng.integers(0, 2**31))
            rate = 0.1 + rng.uniform(-0.03, 0.03)
            sigmas = [max(0.12, 0.6 * np.exp(-rate * s)) for s in range(n_sessions)]
            lapses = [max(0.02, 0.15 * np.exp(-0.12 * s)) for s in range(n_sessions)]
            per_session_kwargs = [{'sigma': sigmas[s], 'lapse': lapses[s]}
                                  for s in range(n_sessions)]
            dist_schedule = (['uniform'] * shift_session
                             + ['exponential_left'] * (n_sessions - shift_session))
            animal, _ = generate_synthetic_animal(
                animal_id=animal_id, n_sessions=n_sessions, trials_per_session=300,
                seed=animal_seed, simulator=_learning_simulator,
                per_session_simulator_kwargs=per_session_kwargs,
                distribution_schedule=dist_schedule, stage='Full_Task_Cont',
            )
            experiment.add_animal(animal)
        return experiment

    experiment = generate_cohort()
    SHIFT_SESSION = 15
    STAGE = 'Full_Task_Cont'
    print(f"Generated synthetic cohort: {experiment.n_animals} animals, {experiment.animal_ids}")
else:
    SHIFT_SESSION = None
    STAGE = 'Full_Task_Cont'
    print(f"Loaded: {experiment.n_animals} animals, {experiment.animal_ids}")

---
## 2. Data Hierarchy

```
ExperimentData
└── AnimalData  (one per mouse)
    └── SessionData  (one per day)
        ├── SessionMetadata  (stage, contingency, stim range, ...)
        └── TrialData  (stimulus, choice, outcome, RT, opto, ...)
```

In [ ]:
# Experiment
print("=== Experiment Summary ===")
print(experiment.summary().to_string(index=False))
print()

# Animal
animal = experiment.get_animal(experiment.animal_ids[0])
print(f"=== Animal: {animal.animal_id} ===")
print(f"  Sessions: {animal.n_sessions}, Stages: {animal.stages}")
print(f"  Dates: {animal.sessions[0].date} → {animal.sessions[-1].date}")
print()

# Session
session = animal.sessions[0]
print(f"=== Session: {session.session_id} ===")
print(f"  Stage: {session.stage}, Distribution: {session.distribution}")
print(f"  Trials: {session.n_trials} total, {session.trials.valid_mask.sum()} valid, "
      f"{session.trials.abort.sum()} abort")

# Trial-level
trials = session.trials
print(f"\n=== TrialData ===")
print(f"  stimulus: [{trials.stimulus.min():.2f}, {trials.stimulus.max():.2f}]")
print(f"  choice unique: {np.unique(trials.choice[~np.isnan(trials.choice)])}")
print(f"  opto_on: {trials.opto_on.sum()} trials")

## 3. Session-Level Summaries

In [ ]:
rows = [s.summary() for s in animal.sessions]
session_df = pd.DataFrame(rows)
session_df['perf'] = session_df['perf'].round(3)
print(f"Sessions for {animal.animal_id}:")
print(session_df[['session_id', 'session_idx', 'date', 'stage',
                   'distribution', 'n_trials', 'n_valid', 'perf']].to_string(index=False))

## 4. Summary Statistics

**DEFAULT_STATS** : accuracy, psychometric, recency, win_stay, stimulus_sensitivity  
**FEATURE_MATRIX_STATS** : full set including entropy, perseveration, update matrix, etc.

In [ ]:
session_early = animal.sessions[1]
session_late = animal.sessions[-1]

early_s = session_early.stats(stat_names=['accuracy', 'psychometric', 'recency'])
late_s = session_late.stats(stat_names=['accuracy', 'psychometric', 'recency'])

header = f"{'Metric':<20} {'Early (S1)':<15} {'Late (S' + str(session_late.session_idx) + ')':<15}"
print(header)
print("-" * len(header))
print(f"{'accuracy':<20} {early_s['accuracy']:<15.3f} {late_s['accuracy']:<15.3f}")
for key in ['pse', 'slope', 'lapse_low', 'lapse_high']:
    ev = early_s['psychometric'].get(key, np.nan)
    lv = late_s['psychometric'].get(key, np.nan)
    print(f"{key:<20} {f'{ev:.3f}':<15} {f'{lv:.3f}':<15}")
print(f"{'recency':<20} {early_s['recency']:<15.3f} {late_s['recency']:<15.3f}")

## 5. Psychometric Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (idx, label) in zip(axes, [(1, 'naive'), (10, 'learning'), (-1, 'expert/post-shift')]):
    s = animal.sessions[idx]
    arrays = s.trials.get_arrays()
    valid = ~arrays['no_response']
    plot_psychometric(arrays['stimuli'][valid], arrays['choices'][valid],
                      ax=ax, title=f'{s.session_id} ({label})')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_session_psychometrics(animal.sessions[:5], mode='overlay', ax=axes[0])
axes[0].set_title('First 5 sessions')
plot_session_psychometrics(animal.sessions[-5:], mode='overlay', ax=axes[1])
axes[1].set_title('Last 5 sessions')
plt.tight_layout()
plt.show()

## 6. Feature Matrix

In [ ]:
df = build_feature_matrix(animal, stage=STAGE)
print(f"Feature matrix: {df.shape[0]} sessions × {df.shape[1]} columns")
print(f"\nColumns ({len(df.columns)}):")
for col in sorted(df.columns):
    print(f"  {col}")

## 7. Stat Trajectories

Using `behav_utils.plotting.trajectory`: `plot_stat_trajectory` (single animal), `plot_multi_animal_trajectory` (group), `plot_stat_grid` (multi-panel grid).

In [ ]:
# Single animal: one stat
fig, ax = animal.plot_trajectory('accuracy', stage=STAGE)
if SHIFT_SESSION is not None:
    ax.axvline(SHIFT_SESSION, color='red', linestyle='--', alpha=0.6, label='shift')
    ax.legend(fontsize=8)
plt.show()

In [ ]:
# Single animal: stat grid
stats_to_plot = ['accuracy', 'pse', 'slope', 'recency',
                 'side_bias', 'choice_entropy', 'win_stay', 'lapse_low']
stats_to_plot = [s for s in stats_to_plot if s in df.columns]

fig, axes = plot_stat_grid(
    [animal], stats=stats_to_plot, stage=STAGE,
    combine='none', show_individual=True,
)
if SHIFT_SESSION is not None:
    for ax in axes.flatten():
        if ax.get_visible():
            ax.axvline(SHIFT_SESSION, color='red', linestyle='--', alpha=0.4)
plt.show()

In [ ]:
# Multi-animal: mean ± SEM
all_animals = experiment.get_animals(min_sessions=5)

fig, axes = plot_stat_grid(
    all_animals, stats=stats_to_plot, stage=STAGE,
    combine='mean_sem', show_individual=True, individual_alpha=0.2,
)
if SHIFT_SESSION is not None:
    for ax in axes.flatten():
        if ax.get_visible():
            ax.axvline(SHIFT_SESSION, color='red', linestyle='--', alpha=0.4)
plt.show()

---
## 8. Update Matrices & Serial Dependence

The update matrix captures serial dependence: how does the previous trial's
stimulus shift the current psychometric curve? This is the critical link
between model predictions and actual behaviour.

### 8.1 Naive vs Expert

In [ ]:
# Compare update matrix: early (naive) vs late (expert) session
sessions = animal.get_sessions(stage=STAGE)
EARLY_IDX = min(2, len(sessions) - 1)
LATE_IDX = len(sessions) - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for col, (sidx, label) in enumerate([(EARLY_IDX, 'Naive'), (LATE_IDX, 'Expert')]):
    sess = sessions[sidx]
    arrays = sess.trials.get_arrays(exclude_abort=True, exclude_opto=True)
    valid = ~arrays['no_response']
    s, c, cat = arrays['stimuli'][valid], arrays['choices'][valid], arrays['categories'][valid]

    um, cond_mat, info = compute_update_matrix(s, c, cat, n_bins=8)
    plot_update_matrix(um, title=f'{label} (session {sidx})', ax=axes[col])

plt.tight_layout()
plt.show()

### 8.2 Update matrix evolution across learning

In [ ]:
# Compute update matrices at several points across learning
sessions = animal.get_sessions(stage=STAGE)
n_sess = len(sessions)
indices = np.linspace(0, n_sess - 1, min(5, n_sess), dtype=int)
indices = sorted(set(indices))

fig, axes = plt.subplots(1, len(indices), figsize=(3.5 * len(indices), 3.5))
if len(indices) == 1:
    axes = [axes]

# Compute all matrices first for consistent colour scale
ums = []
for sidx in indices:
    sess = sessions[sidx]
    arrays = sess.trials.get_arrays(exclude_abort=True, exclude_opto=True)
    valid = ~arrays['no_response']
    s, c, cat = arrays['stimuli'][valid], arrays['choices'][valid], arrays['categories'][valid]
    um, _, info = compute_update_matrix(s, c, cat, n_bins=8)
    ums.append(um)

vmax = max(np.nanmax(np.abs(um)) for um in ums if not np.all(np.isnan(um)))

for i, (sidx, um) in enumerate(zip(indices, ums)):
    plot_update_matrix(um, title=f'S{sidx}', ax=axes[i], vmin=-vmax, vmax=vmax)

fig.suptitle(f'{animal.animal_id}: Update matrix evolution', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 8.3 Serial dependence profiles

Overlay the column-mean of the update matrix (serial dependence profile) across sessions. As the animal learns, the profile should flatten (serial dependence diminishes).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cmap_sess = plt.cm.viridis

midpoints = np.linspace(-1, 1, 9)
midpoints = (midpoints[:-1] + midpoints[1:]) / 2

for i, (sidx, um) in enumerate(zip(indices, ums)):
    profile = np.nanmean(um, axis=0)
    frac = i / max(len(indices) - 1, 1)
    ax.plot(midpoints, profile, 'o-', color=cmap_sess(frac), lw=1.5, ms=5,
            label=f'S{sidx}', alpha=0.8)

ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Previous stimulus bin')
ax.set_ylabel('Mean P(B) shift')
ax.set_title(f'{animal.animal_id}: Serial dependence profile evolution')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 8.4 Conditional psychometric curves

For an expert session, plot psychometric curves conditioned on the previous stimulus bin.

In [ ]:
sess = sessions[10]
arrays = sess.trials.get_arrays(exclude_abort=True, exclude_opto=True)
valid = ~arrays['no_response']
s, c, cat = arrays['stimuli'][valid], arrays['choices'][valid], arrays['categories'][valid]

um, cond_mat, info = compute_update_matrix(s, c, cat, n_bins=8)

fig, ax = plt.subplots(figsize=(7, 5))
x_fine = np.linspace(-1, 1, 200)
n_bins = 8
bin_edges = np.linspace(-1, 1, n_bins + 1)
midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
cmap = plt.cm.coolwarm

for b in range(n_bins):
    row = cond_mat[b]
    if row is None or len(row) == 0:
        continue
    # row is a dict from compute_update_matrix with psychometric params per previous bin
    if isinstance(row, dict) and 'mu' in row and not np.isnan(row['mu']):
        y = cumulative_gaussian(x_fine, row['mu'], row['sigma'],
                                row.get('lapse_low', 0), row.get('lapse_high', 0))
        frac = b / (n_bins - 1)
        ax.plot(x_fine, y, color=cmap(frac), lw=1.5,
                label=f'Prev bin {midpoints[b]:.2f}', alpha=0.8)

ax.axvline(0, color='grey', lw=0.5, ls='--', alpha=0.3)
ax.axhline(0.5, color='grey', lw=0.5, ls='--', alpha=0.3)
ax.set_xlabel('Current stimulus')
ax.set_ylabel('P(choose B)')
ax.set_title(f'{sess.session_id}: Conditional psychometric curves')
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 9. Saving & Reloading

```python
experiment.save('experiment.pkl')
animal.save('test.pkl')

# Reload:
experiment = ExperimentData.load('experiment.pkl')
animal = AnimalData.load('test.pkl')
```